# Gurumoji P0 A100 transport smoke

This notebook contains one synthetic job and a small worker source bundle. It has no user documents, credentials, or evaluation answers.

Run all cells in order on an A100 runtime. The first cell checks the GPU and creates one synthetic P0 result; it does not call an LLM or process papers. The second cell asks for Drive authorization, saves the result ZIP directly to the shared folder, then starts scheduled monitoring.

While the runtime remains available, it checks Runtime/GPU state and the job folder every 60 minutes; when an A100 is assigned, it runs one lightweight 256x256 FP16 health check. It verifies Drive artifact file IDs, sizes, and SHA-256 every 80 minutes. It updates one monitor JSON file, resumes recent history after a restart, and ends after at most 24 hours per run.

The returned ZIP is still untrusted and is not accepted into local JobStore. The current prototype has no production model adapter: future job bundles are detected for manual review, and the only hourly GPU work without a job is the small A100 health check. Colab may disconnect earlier under its service limits; this schedule cannot guarantee a continuous connection.


In [ ]:
import base64
import hashlib
import json
import sys
import zipfile
from datetime import datetime, timezone
from io import BytesIO
from pathlib import Path, PurePosixPath

import torch

BUNDLE_B64 = """..."""
EXPECTED_BUNDLE_SHA256 = "ad4041cee5600623ac5e9ca5a6855b3b9653266266e6de429d8626773c7c0415"
EXPECTED_JOB_ID = "p0-colab-roundtrip-c11a7321bee8"
EXPECTED_GENERATION = 1

def sha256(data):
    return hashlib.sha256(data).hexdigest()

bundle_bytes = base64.b64decode(BUNDLE_B64, validate=True)
assert sha256(bundle_bytes) == EXPECTED_BUNDLE_SHA256, "P0 bundle checksum mismatch"
with zipfile.ZipFile(BytesIO(bundle_bytes)) as zf:
    expected_members = {"request.zip", "approved_sources.json", "worker_sources.zip", "bundle_manifest.json"}
    assert set(zf.namelist()) == expected_members, "Unexpected bundle files"
    bundle = {name: zf.read(name) for name in expected_members}

bundle_manifest = json.loads(bundle["bundle_manifest.json"])
assert bundle_manifest["job_id"] == EXPECTED_JOB_ID
assert bundle_manifest["generation"] == EXPECTED_GENERATION
for name, digest in bundle_manifest["files"].items():
    assert name in bundle and sha256(bundle[name]) == digest, f"Bundle member hash mismatch: {name}"

# Validate the bundled worker source file set and every source hash before import.
worker_root = Path("/content/p0_worker")
worker_root.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BytesIO(bundle["worker_sources.zip"])) as zf:
    names = set(zf.namelist())
    assert "worker_sources_manifest.json" in names
    manifest = json.loads(zf.read("worker_sources_manifest.json"))
    source_hashes = manifest["files"]
    assert names == set(source_hashes) | {"worker_sources_manifest.json"}
    for name, digest in source_hashes.items():
        member = PurePosixPath(name)
        assert not member.is_absolute() and ".." not in member.parts
        content = zf.read(name)
        assert sha256(content) == digest, f"Worker source hash mismatch: {name}"
        destination = worker_root.joinpath(*member.parts)
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(content)
sys.path.insert(0, str(worker_root / "src"))

from gurumoji.knowledge_builder import contracts, transport
from gurumoji.knowledge_builder.colab_worker import run_colab_worker

registry_value = json.loads(bundle["approved_sources.json"])
assert registry_value["schema_version"] == 1
approved_sources = {source["source_id"]: source for source in registry_value["sources"]}
request_archive, members = transport._read_safe_zip(bundle["request.zip"])
try:
    handoff = transport._parse_canonical_json(transport._zip_read(request_archive, members, "handoff.json"))
    request_data = transport._zip_read(request_archive, members, "request.json")
    request = transport._parse_canonical_json(request_data)
finally:
    request_archive.close()
job = handoff["job"]
assert job["job_id"] == EXPECTED_JOB_ID and job["generation"] == EXPECTED_GENERATION
assert sha256(request_data) == job["input_manifest_sha256"]

# This small CUDA operation confirms the selected A100 is live. The generator
# below is deterministic fixture code, not an LLM call or analytical claim.
assert torch.cuda.is_available(), "CUDA GPU is required; select an A100 runtime"
device_name = torch.cuda.get_device_name(0)
assert "A100" in device_name.upper(), f"Expected A100, found {device_name}"
left = torch.ones((512, 512), device="cuda")
right = torch.ones((512, 512), device="cuda")
product = left @ right
torch.cuda.synchronize()
gpu_first = float(product[0, 0].item())
assert gpu_first == 512.0

def generate(task, excerpts):
    assert task["expert_id"] == "exp-p0-transport-fixture"
    assert len(excerpts) == 1
    excerpt = excerpts[0]
    source = approved_sources[excerpt["source_id"]]
    now = datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")
    claim = {
        "schema_version": 1,
        "claim_id": "claim-p0-transport-fixture",
        "expert_id": task["expert_id"],
        "claim": "Synthetic transport fixture only; this is not method guidance.",
        "applicability": ["P0 transport smoke test"],
        "exceptions": [],
        "claim_type": "literature",
        "evidence": [{
            "source_id": source["source_id"], "source_version": source["version"],
            "source_sha256": source["sha256"], "anchor_id": excerpt["anchor_id"],
            "span": {"kind": "page", "start": "1", "end": "1"},
            "extracted_text_sha256": excerpt["extracted_text_sha256"], "transform_history": [],
        }],
        "contradictions": [],
        "status": "candidate",
        "provenance": {"method": "deterministic P0 transport fixture", "created_at": now},
        "rights": source["rights"],
        "reviewer": {"identity": "synthetic-fixture", "reviewed_at": now, "target_sha256": "0" * 64},
    }
    claim["reviewer"]["target_sha256"] = contracts.claim_review_target(claim)
    return [claim]

result_zip = run_colab_worker(bundle["request.zip"], approved_sources=approved_sources, generate=generate)
result_archive, result_members = transport._read_safe_zip(result_zip)
try:
    result_manifest = transport._parse_canonical_json(
        transport._zip_read(result_archive, result_members, transport._RESULT_MANIFEST))
finally:
    result_archive.close()
commit = result_manifest["commit"]
summary = {
    "status": "worker_result_created_unaccepted",
    "job_id": job["job_id"], "generation": job["generation"],
    "request_zip_sha256": sha256(bundle["request.zip"]),
    "input_manifest_sha256": job["input_manifest_sha256"],
    "result_zip_sha256": sha256(result_zip), "result_zip_size_bytes": len(result_zip),
    "commit_manifest_sha256": commit["manifest_sha256"],
    "artifact_count": len(commit["artifacts"]),
    "gpu": device_name, "gpu_matrix_first": gpu_first,
    "generator": "deterministic fixture; no LLM inference",
}
print("P0_RESULT_JSON=" + json.dumps(summary, sort_keys=True))


## A100 periodic checks and Drive artifact audit

Run all cells in order. The first cell checks the A100 and creates one synthetic P0 result. The second cell saves that ZIP to the shared Drive folder, then keeps a monitor cell running for up to 24 hours per run.

- Every 60 minutes it checks Runtime/GPU state and queued job candidates. When an A100 is assigned, it runs one small 256x256 FP16 matrix multiplication as a health check.
- Every 80 minutes it checks the tracked Drive bundle/result file IDs, sizes, and SHA-256. It saves the latest monitor history in one JSON file and resumes that history after a restart.
- The hourly GPU check is only a tiny health probe. It does not generate dummy documents or claims. The current prototype has no production model adapter, so queued real jobs are recorded for manual review rather than processed.

The monitor stays in execution between checks, but Colab may terminate the runtime earlier under its service limits. This schedule cannot guarantee a continuous connection. The result ZIP remains untrusted until the local acceptance script validates it.


In [ ]:
# Gurumoji A100 runtime scheduler
import hashlib
import json
import time
from datetime import datetime, timezone
from io import BytesIO

import google.auth
from google.colab import auth as colab_auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseUpload

DRIVE_FOLDER_ID = "1wd69nc_9Xaw80Id54Nt7WW_9x3ztm9j5"
EXPECTED_BUNDLE_FILE_ID = "1WkhBlBxrIKZVMEwQk_Gcm7vTVS0upHWw"
EXPECTED_BUNDLE_NAME = "Gurumoji_P0_Colab_Roundtrip_Bundle.zip"
EXPECTED_BUNDLE_SHA256 = "ad4041cee5600623ac5e9ca5a6855b3b9653266266e6de429d8626773c7c0415"
EXPECTED_BUNDLE_SIZE = 27358
EXPECTED_JOB_ID = "p0-colab-roundtrip-c11a7321bee8"
EXPECTED_GENERATION = 1
STATUS_INTERVAL_SECONDS = 60 * 60
DRIVE_AUDIT_INTERVAL_SECONDS = 80 * 60
MAX_MONITOR_HOURS = 24
MONITOR_FILE_NAME = f"Gurumoji_A100_Runtime_Monitor_{EXPECTED_JOB_ID}_g{EXPECTED_GENERATION}.json"
RESULT_FILE_NAME = f"Gurumoji_P0_Result_{EXPECTED_JOB_ID}_g{EXPECTED_GENERATION}.zip"

colab_auth.authenticate_user()
credentials, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive"])
drive_api = build("drive", "v3", credentials=credentials, cache_discovery=False)

def drive_list_folder():
    files = []
    page_token = None
    while True:
        response = drive_api.files().list(
            q=f"'{DRIVE_FOLDER_ID}' in parents and trashed = false",
            pageSize=1000,
            pageToken=page_token,
            fields="nextPageToken,files(id,name,size,md5Checksum,mimeType,parents,modifiedTime)",
            supportsAllDrives=True,
            includeItemsFromAllDrives=True,
        ).execute(num_retries=3)
        files.extend(response.get("files", []))
        page_token = response.get("nextPageToken")
        if not page_token:
            return files

def drive_read(file_id):
    return drive_api.files().get_media(fileId=file_id, supportsAllDrives=True).execute(num_retries=3)

def drive_upload_or_reuse(name, content, mime_type):
    digest = hashlib.sha256(content).hexdigest()
    matches = [item for item in drive_list_folder() if item["name"] == name]
    if len(matches) > 1:
        raise RuntimeError(f"Multiple Drive files have the reserved name {name!r}")
    if matches:
        existing = matches[0]
        existing_bytes = drive_read(existing["id"])
        if name == RESULT_FILE_NAME:
            archive, members = transport._read_safe_zip(existing_bytes)
            try:
                remote = transport._parse_canonical_json(
                    transport._zip_read(archive, members, transport._RESULT_MANIFEST))
                result_job = remote["job"]
                commit = remote["commit"]
                if (remote.get("kind") != "colab_result" or
                        result_job.get("job_id") != EXPECTED_JOB_ID or
                        result_job.get("generation") != EXPECTED_GENERATION or
                        result_job.get("input_manifest_sha256") != job["input_manifest_sha256"] or
                        commit.get("job_id") != EXPECTED_JOB_ID or
                        commit.get("generation") != EXPECTED_GENERATION):
                    raise RuntimeError("Existing Drive result belongs to a different job generation")
            finally:
                archive.close()
        existing_sha256 = hashlib.sha256(existing_bytes).hexdigest()
        if name == RESULT_FILE_NAME and existing_sha256 != digest:
            prior_monitors = [item for item in drive_list_folder()
                              if item["name"] == MONITOR_FILE_NAME]
            if len(prior_monitors) != 1:
                raise RuntimeError("Existing Drive result differs from this run and has no unique integrity pin")
            prior_monitor = json.loads(drive_read(prior_monitors[0]["id"]).decode("utf-8"))
            if (prior_monitor.get("schema_version") != 1 or
                    prior_monitor.get("job_id") != EXPECTED_JOB_ID or
                    prior_monitor.get("generation") != EXPECTED_GENERATION):
                raise RuntimeError("Existing Drive result integrity pin belongs to another job generation")
            prior_result_pins = [item for item in prior_monitor.get("tracked_artifacts", [])
                                 if item.get("id") == existing["id"] and
                                 item.get("name") == RESULT_FILE_NAME]
            if (len(prior_result_pins) != 1 or
                    prior_result_pins[0].get("sha256") != existing_sha256 or
                    prior_result_pins[0].get("size_bytes") != len(existing_bytes)):
                raise RuntimeError("Existing Drive result differs from its persisted integrity pin")
            return existing, existing_bytes
        if existing_sha256 != digest:
            raise RuntimeError(f"Refusing to overwrite a different existing artifact: {name}")
        return existing, existing_bytes
    media = MediaIoBaseUpload(BytesIO(content), mimetype=mime_type, resumable=True)
    created = drive_api.files().create(
        body={"name": name, "mimeType": mime_type, "parents": [DRIVE_FOLDER_ID]},
        media_body=media,
        fields="id,name,size,md5Checksum,mimeType,parents,modifiedTime",
        supportsAllDrives=True,
    ).execute(num_retries=3)
    return created, content

# The synthetic P0 result is written to Drive once. It remains untrusted until
# the local acceptance script validates it and advances the local generation.
result_drive_file, saved_result_zip = drive_upload_or_reuse(
    RESULT_FILE_NAME, result_zip, "application/zip")
result_zip = saved_result_zip
result_sha256 = sha256(result_zip)
tracked_artifacts = [
    {
        "id": EXPECTED_BUNDLE_FILE_ID,
        "name": EXPECTED_BUNDLE_NAME,
        "size_bytes": EXPECTED_BUNDLE_SIZE,
        "sha256": EXPECTED_BUNDLE_SHA256,
    },
    {
        "id": result_drive_file["id"],
        "name": RESULT_FILE_NAME,
        "size_bytes": len(result_zip),
        "sha256": result_sha256,
    },
]

def utc_now():
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")

def a100_health_probe(gpu):
    if not torch.cuda.is_available() or "A100" not in gpu.upper():
        return {"status": "skipped", "reason": "a100_not_available"}
    try:
        left = torch.ones((256, 256), device="cuda", dtype=torch.float16)
        right = torch.ones((256, 256), device="cuda", dtype=torch.float16)
        product = left @ right
        torch.cuda.synchronize()
        result = float(product[0, 0].item())
        del left, right, product
    except Exception as exc:
        return {"status": "failed", "error_type": type(exc).__name__}
    if result != 256.0:
        return {"status": "failed", "operation": "256x256_fp16_matmul", "result": result}
    return {"status": "passed", "operation": "256x256_fp16_matmul", "result": result}

def runtime_status(files):
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
    if torch.cuda.is_available():
        free_bytes, total_bytes = torch.cuda.mem_get_info()
        gpu_memory = {"free_mib": round(free_bytes / (1024 * 1024)),
                      "total_mib": round(total_bytes / (1024 * 1024))}
    else:
        gpu_memory = None
    queued = sorted(
        item["name"] for item in files
        if item["name"].startswith("Gurumoji_Job_") and item["name"].endswith(".zip")
    )
    gpu_health_probe = a100_health_probe(gpu)
    return {
        "checked_at": utc_now(),
        "event": "hourly_runtime_and_queue_check",
        "gpu": gpu,
        "is_a100": "A100" in gpu.upper(),
        "gpu_memory": gpu_memory,
        "queued_job_bundles": queued,
        "queue_action": "no_job_waiting" if not queued else "manual_review_required_no_production_adapter",
        "gpu_health_probe": gpu_health_probe,
        "gpu_work": "a100_health_probe_no_production_adapter" if gpu_health_probe["status"] != "skipped" else "none_no_a100_assignment",
    }

def audit_drive_artifacts():
    files_by_id = {item["id"]: item for item in drive_list_folder()}
    checked = []
    for expected in tracked_artifacts:
        item = files_by_id.get(expected["id"])
        if item is None:
            raise RuntimeError(f"Expected Drive artifact is missing: {expected['name']}")
        if item["name"] != expected["name"]:
            raise RuntimeError(f"Drive artifact name changed: {expected['id']}")
        if DRIVE_FOLDER_ID not in item.get("parents", []):
            raise RuntimeError(f"Drive artifact moved out of the configured folder: {expected['name']}")
        content = drive_read(expected["id"])
        actual_size = len(content)
        actual_sha256 = hashlib.sha256(content).hexdigest()
        if actual_size != expected["size_bytes"] or actual_sha256 != expected["sha256"]:
            raise RuntimeError(f"Drive artifact verification failed: {expected['name']}")
        checked.append({"id": expected["id"], "name": expected["name"],
                        "size_bytes": actual_size, "sha256": actual_sha256})
    return {"checked_at": utc_now(), "event": "80_minute_drive_artifact_audit",
            "status": "verified", "artifacts": checked}

def save_monitor_state(events):
    global monitor_drive_file_id
    payload = json.dumps({
        "schema_version": 1,
        "job_id": EXPECTED_JOB_ID,
        "generation": EXPECTED_GENERATION,
        "session_started_at": monitor_started_at,
        "monitor_interval_minutes": 60,
        "drive_audit_interval_minutes": 80,
        "max_monitor_hours": MAX_MONITOR_HOURS,
        "tracked_artifacts": tracked_artifacts,
        "events": events[-100:],
    }, ensure_ascii=False, sort_keys=True, indent=2).encode("utf-8")
    media = MediaIoBaseUpload(BytesIO(payload), mimetype="application/json", resumable=True)
    if monitor_drive_file_id is None:
        response = drive_api.files().create(
            body={"name": MONITOR_FILE_NAME, "mimeType": "application/json",
                  "parents": [DRIVE_FOLDER_ID]},
            media_body=media,
            fields="id,name,size,parents,modifiedTime",
            supportsAllDrives=True,
        ).execute(num_retries=3)
        monitor_drive_file_id = response["id"]
    else:
        response = drive_api.files().update(
            fileId=monitor_drive_file_id,
            media_body=media,
            fields="id,name,size,parents,modifiedTime",
            supportsAllDrives=True,
        ).execute(num_retries=3)
    if DRIVE_FOLDER_ID not in response.get("parents", [DRIVE_FOLDER_ID]):
        raise RuntimeError("Runtime monitor file was not saved in the configured Drive folder")

monitor_drive_file_id = None
events = []
existing_monitors = [item for item in drive_list_folder() if item["name"] == MONITOR_FILE_NAME]
if len(existing_monitors) > 1:
    raise RuntimeError("Multiple runtime monitor files exist in the configured Drive folder")
if existing_monitors:
    previous = json.loads(drive_read(existing_monitors[0]["id"]).decode("utf-8"))
    if (previous.get("schema_version") != 1 or
            previous.get("job_id") != EXPECTED_JOB_ID or
            previous.get("generation") != EXPECTED_GENERATION):
        raise RuntimeError("Refusing to resume an unrelated Drive monitor file")
    prior_events = previous.get("events", [])
    if not isinstance(prior_events, list) or any(not isinstance(item, dict) for item in prior_events):
        raise RuntimeError("Existing Drive monitor history has an invalid shape")
    events.extend(prior_events[-100:])
    monitor_drive_file_id = existing_monitors[0]["id"]
monitor_started_at = utc_now()
monitor_started = time.monotonic()
stop_at = monitor_started + MAX_MONITOR_HOURS * 60 * 60
next_status = monitor_started
next_drive_audit = monitor_started
print(f"Monitor started at {monitor_started_at}; 60-minute status checks, "
      f"80-minute Drive hash audits, no synthetic idle GPU load, "
      f"maximum {MAX_MONITOR_HOURS} hours. Existing monitor history: {len(events)} events.")
print(f"P0 result saved to Drive: {result_drive_file['id']} ({result_sha256})")

try:
    while time.monotonic() < stop_at:
        now = time.monotonic()
        files = None
        if now >= next_status:
            files = drive_list_folder()
            event = runtime_status(files)
            events.append(event)
            print("RUNTIME_STATUS=" + json.dumps(event, ensure_ascii=False, sort_keys=True))
            next_status += STATUS_INTERVAL_SECONDS
        if now >= next_drive_audit:
            audit = audit_drive_artifacts()
            events.append(audit)
            print("DRIVE_AUDIT=" + json.dumps(audit, ensure_ascii=False, sort_keys=True))
            next_drive_audit += DRIVE_AUDIT_INTERVAL_SECONDS
        if events:
            save_monitor_state(events)
        next_check = min(next_status, next_drive_audit, stop_at)
        time.sleep(max(1, next_check - time.monotonic()))
except KeyboardInterrupt:
    print("Monitor stopped by user; latest state was saved to Drive.")
finally:
    save_monitor_state(events)
print("Monitor window ended. Colab may also stop a runtime earlier under its service limits.")


## A100 periodic checks and Drive artifact audit

Run all cells in order. The first cell must finish before this cell starts. It saves the synthetic result ZIP to the shared Drive folder, then checks runtime/GPU availability and job candidates every 60 minutes. It verifies bundle/result file IDs, sizes, and SHA-256 every 80 minutes. One monitor JSON file is updated in place and recent history is resumed after a restart.

The monitor ends after at most 24 hours per run. When the job queue is empty, it runs a small 256x256 FP16 A100 health check once per hour if an A100 is assigned, then records status and checks Drive. Future job bundles are detected and recorded but require a production model adapter that is not implemented yet. Colab may terminate the runtime earlier under its service limits, so this cannot guarantee a continuous connection.
